# House Prices Dataset
[kaggle](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact
from IPython.display import Markdown
from tqdm import tqdm
%matplotlib inline

In [2]:
import torch
from torch import nn

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

In [4]:
df = pd.read_csv(path + '/train.csv')
df_test = pd.read_csv(path + '/test.csv')

In [5]:
input_columns = {col for col in df.columns if df[col].dtype.kind in ('i', 'f')} - {'Id', 'SalePrice'}
df = df[list(input_columns) + ['SalePrice']].dropna()
X = df[sorted(list(input_columns))]
y = df[['SalePrice']]

In [6]:
# display(y, X)

In [7]:
A, B = X.mean(), X.std()
C, D = y.mean(), y.std()

X_normalized = (X - A) / B
y_normalized = (y - C) / D

In [8]:
# A, B, C, D
X_normalized.shape, y_normalized.shape

((1121, 36), (1121, 1))

In [9]:
X_tensor = torch.tensor(X_normalized.values, dtype=torch.float32)
y_tensor = torch.tensor(y_normalized.values, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(len(input_columns), 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)
opt = torch.optim.SGD(model.parameters(), lr=0.01)

for _ in tqdm(range(50_000)):
    loss = nn.functional.mse_loss(model(X_tensor), y_tensor)
    opt.zero_grad()
    loss.backward()
    opt.step()

    with torch.no_grad():
        list(model)[0].weight.clamp_(min=0)
        list(model)[2].weight.clamp_(min=0)

100%|██████████| 50000/50000 [00:19<00:00, 2577.24it/s]


In [10]:
SHOW_ALL_PARAMETERS = False # or True
for layer in model:
    if SHOW_ALL_PARAMETERS and isinstance(layer, nn.Linear):
        display(layer.weight, layer.bias)

In [11]:
y_pred = model(X_tensor)

mae = (y_tensor - y_pred).abs().mean()
print(f'Mean Absolute Error: {mae:.2f}')

mse = ((y_tensor - y_pred) ** 2).mean()
print(f'Mean Squared Error: {mse:.2f}')

Mean Absolute Error: 0.21
Mean Squared Error: 0.14


In [12]:
weights_df = pd.DataFrame(sorted([
    {'Column': col, 'Weight': float(weights.sum())}
    for col, weights in zip(X.columns, list(model)[0].weight.detach().T)
], key=lambda x: x['Weight'], reverse=True)[:20])

display(Markdown('## Najbardziej znaczące kolumny dla modelu'))
weights_df

## Najbardziej znaczące kolumny dla modelu

,Column,Weight
0,OverallQual,1.497636
1,GarageCars,0.782132
2,OverallCond,0.747537
3,ScreenPorch,0.713920
4,Fireplaces,0.641420
5,FullBath,0.616674
6,BsmtFullBath,0.583565
7,YearBuilt,0.572376
8,YrSold,0.500320
9,GarageYrBlt,0.483304
